<center><h1>Zhu_Lingfei_HW3</h1></center>
<br>
<br>

Name: Zhu Lingfei
<br>
Github Username: youth0505
<br>
USC ID: 8599420339

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

SENSOR_COLUMNS = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
STATS = ["min", "max", "mean", "median", "std", "q1", "q3"]

def read_file(path):
    return pd.read_csv(path, comment="#", header=None,
                       names=["time"] + SENSOR_COLUMNS,
                       sep=r"\s*,\s*|\s+", engine="python",
                       usecols=range(7))[SENSOR_COLUMNS]

def get_features(data):
    result = {}
    for col in SENSOR_COLUMNS:
        x = data[col]
        result.update({f"min_{col}": x.min(), f"max_{col}": x.max(),
                       f"mean_{col}": x.mean(), f"median_{col}": x.median(),
                       f"std_{col}": x.std(), f"q1_{col}": x.quantile(.25),
                       f"q3_{col}": x.quantile(.75)})
    return result

Get the AReM Data Set

In [2]:
DATA_DIR = Path("../Data/AReM")
if not DATA_DIR.exists():
    DATA_DIR = Path("Data/AReM")
files = sorted(DATA_DIR.glob("*/*.csv"))
data = {path: read_file(path) for path in files}
print(f"Loaded {len(files)} files")
print("Rows per file:", pd.Series([len(frame) for frame in data.values()]).value_counts().to_dict())

Loaded 88 files
Rows per file: {480: 87, 479: 1}


### (b) Test and Train Data

In [3]:
test_files, train_files = [], []
for path in files:
    number = int(path.stem.replace("dataset", ""))
    test = (path.parent.name.startswith("bending") and number in [1, 2])
    test |= (not path.parent.name.startswith("bending") and number in [1, 2, 3])
    (test_files if test else train_files).append(path)

print(f"Training: {len(train_files)}; test: {len(test_files)}")

Training: 69; test: 19


### (c) Feature Extraction

#### i. Research

Time-domain features commonly used in time-series classification include location (mean, median, trimmed mean), spread (variance, standard deviation, interquartile range, median absolute deviation), extrema and range, quantiles, root-mean-square energy, zero-crossing rate, and signal slope or trend. This assignment requests minimum, maximum, mean, median, standard deviation, and the first and third quartiles.

#### ii. Extraction

In [4]:
def make_features(paths):
    rows = []
    for path in paths:
        row = get_features(data[path])
        row["activity"] = "bending" if path.parent.name.startswith("bending") else path.parent.name
        row["instance"] = f"{path.parent.name}/{path.stem}"
        rows.append(row)
    frame = pd.DataFrame(rows).set_index("instance")
    columns = [f"{stat}_{col}" for col in SENSOR_COLUMNS for stat in STATS]
    return frame[columns + ["activity"]]

features = make_features(files)
train_features = make_features(train_files)
test_features = make_features(test_files)
feature_columns = features.columns[:-1]
print(f"Feature table: {features.shape[0]} instances and {len(feature_columns)} features")
print(features[["min_avg_rss12", "mean_avg_rss12", "max_avg_rss12", "activity"]].head().round(2))

Feature table: 88 instances and 42 features
                   min_avg_rss12  mean_avg_rss12  max_avg_rss12 activity
instance                                                                
bending1/dataset1          37.25           40.62          45.00  bending
bending1/dataset2          38.00           42.81          45.67  bending
bending1/dataset3          35.00           43.95          47.40  bending
bending1/dataset4          33.00           42.18          47.75  bending
bending1/dataset5          33.00           41.68          45.75  bending


#### iii. Standard Deviation

In [5]:
rng = np.random.default_rng(552)
results = {}
for name in feature_columns:
    x = features[name].to_numpy()
    boot_std = [rng.choice(x, len(x), replace=True).std(ddof=1) for _ in range(10000)]
    results[name] = [x.std(ddof=1), *np.quantile(boot_std, [.05, .95])]

std_ci = pd.DataFrame.from_dict(results, orient="index",
                               columns=["std", "90% CI lower", "90% CI upper"])
print(std_ci.round(4))

                     std  90% CI lower  90% CI upper
min_avg_rss12     9.5700        8.2569       10.7417
max_avg_rss12     4.3944        3.3280        5.3082
mean_avg_rss12    5.3357        4.7128        5.8836
median_avg_rss12  5.4401        4.8092        5.9993
std_avg_rss12     1.7722        1.5726        1.9494
q1_avg_rss12      6.1536        5.5706        6.6526
q3_avg_rss12      5.1389        4.3307        5.8409
min_var_rss12     0.0000        0.0000        0.0000
max_var_rss12     5.0627        4.6327        5.4113
mean_var_rss12    1.5742        1.3908        1.7068
median_var_rss12  1.4122        1.2393        1.5428
std_var_rss12     0.8841        0.8032        0.9420
q1_var_rss12      0.9464        0.8282        1.0362
q3_var_rss12      2.1253        1.8991        2.2946
min_avg_rss13     2.9565        2.7599        3.1075
max_avg_rss13     4.8751        4.1626        5.4620
mean_avg_rss13    4.0084        3.4124        4.4867
median_avg_rss13  4.0364        3.4182        

#### iv. Select Features

I chose the minimum, mean, and maximum of avg_rss12. They give a simple summary of the signal's low, typical, and high values.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic model’s training RSS is less than or equal to the linear model’s training RSS. The linear model is nested within the cubic model: the cubic fit can set the quadratic and cubic coefficients to zero and reproduce the linear fit. Adding terms therefore cannot increase the minimized training RSS.

### (b) Linear Test

The linear model is expected to have lower test RSS on average because it matches the true linear relationship and estimates fewer parameters. The cubic model’s extra flexibility can fit training noise and increase variance. A particular finite test sample is not guaranteed to follow this expected ordering.

### (c) Not Linear Train

The cubic model’s training RSS is less than or equal to the linear model’s training RSS, regardless of whether the true relationship is linear. This follows from the cubic model containing the linear model as a special case.

### (d) Not Linear Testing

There is not enough information to determine which test RSS is lower. If the nonlinear relationship is well approximated by a cubic, the cubic model may perform better. If the departure from linearity is small or has a different shape, the cubic model’s extra variance may outweigh its reduction in bias, and the linear model may perform better. 

## 3. ISLR 3.7.3 - Extra Practice 

(a) For fixed GPA \(g\) and IQ, the predicted salary difference between college and high-school graduates is
$$
\hat{Y}_{college}-\hat{Y}_{high}=35-10g
$$
(in thousands of dollars). When \(g>3.5\), this difference is negative, so high-school graduates earn more when GPA is high enough. **iii** is correct.

(b) For a college graduate with GPA \(4.0\) and IQ \(110\):
$$
50+20(4)+0.07(110)+35+0.01(4)(110)-10(4)=137.1
$$
The predicted salary is \$137,100.

(c) False. A small coefficient does not by itself show little evidence of an interaction. Evidence depends on the coefficient’s uncertainty, such as its standard error or p-value.

## 3. ISLR 3.7.5 - Extra Practice 

With no intercept,

$$
\hat{\beta} = \frac{\sum_{j=1}^{n} x_j y_j}{\sum_{k=1}^{n} x_k^2}
$$

Therefore,

$$
\hat{y}_i
= x_i\hat{\beta}
= \sum_{j=1}^{n}
\left(\frac{x_i x_j}{\sum_{k=1}^{n} x_k^2}\right)y_j
$$

Thus,

$$
a_{ij} = \frac{x_i x_j}{\sum_{k=1}^{n} x_k^2}
$$